In [1]:
!pip install -r requirements.txt

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 840.2/840.2 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 939.7/939.7 kB 30.9 MB/s eta 0:00:00
  Created wheel for pronouncing: filename=pronouncing-0.2.0-py2.py3-none-any.whl size=6234 sha256=581ca0c991ecc539d27089edd13a7308662b2c92f5fd00b0ed0e05a6b332e372
  Stored in directory: /root/.cache/pip/wheels/a0/76/15/dfdf38731993cdc4e86fd6d949c70c0e9786cf00073d8114d4
Successfully built pronouncing


In [ ]:
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoConfig, TrainingArguments, Trainer, BertModel, EarlyStoppingCallback
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
from pypinyin import pinyin, Style
import torch
import torch.nn as nn
import numpy as np
import evaluate
import pandas as pd
import os
import pronouncing

In [ ]:
# ============================================================
# DEVICE CONFIGURATION - Set USE_GPU to True if GPU available
# ============================================================
import torch

USE_GPU = False  # Set to True if you have GPU (CUDA/MPS)

# Detect available device
if USE_GPU:
    if torch.cuda.is_available():
        device = "cuda"
        print("✅ GPU (CUDA) detected and enabled")
    elif torch.backends.mps.is_available():
        device = "mps"  # Mac with Apple Silicon
        print("✅ GPU (MPS) detected and enabled")
    else:
        device = "cpu"
        print("⚠️ GPU requested but not available, falling back to CPU")
else:
    device = "cpu"
    print("✅ Using CPU (set USE_GPU=True to enable GPU)")

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device.upper()}")

### Data Preprocessing

In [ ]:
data_path = "./toxic_dataset_cleaned.csv"
df = pd.read_csv(data_path).dropna(subset=['text', 'label'])
df['text'] = df['text'].astype(str)
df['label'] = df['label'].astype(int)

X_train, X_val, y_train, y_val = train_test_split(
    df["text"], df["label"], test_size=0.2, random_state=42, stratify=df["label"]
)

dataset = DatasetDict({
    "train": Dataset.from_pandas(pd.DataFrame({"text": X_train.values, "label": y_train.values}), preserve_index=False),
    "validation": Dataset.from_pandas(pd.DataFrame({"text": X_val.values, "label": y_val.values}), preserve_index=False)
})

## Context Model

### Define Model

In [ ]:
from transformers import AutoTokenizer

model_name = "google-bert/bert-base-chinese"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

# Map the function over the entire dataset in batches
tokenized_datasets = dataset.map(tokenize_function, batched=True)

### Train

In [ ]:
accuracy = evaluate.load("accuracy")
f1_macro = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1_macro": f1_macro.compute(predictions=preds, references=labels, average="macro")["f1"]
    }

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

# 1. Load model with a classification head
# num_labels should match the number of unique classes in your dataset
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# 2. Define training hyperparameters
training_args = TrainingArguments(
    output_dir="./baseline_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5,
    per_device_train_batch_size=16,
    num_train_epochs=15,
    load_best_model_at_end=True,
    weight_decay=0.01,
    metric_for_best_model="f1_macro",
)

# 3. Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=8)],
)

# 4. Start training
trainer.train()

### Inference

In [ ]:
def predict(text):
    # 1. Tokenize
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)

    # 2. Forward Pass
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)

    # 3. Get Label
    logits = outputs.logits
    pred_idx = torch.argmax(logits, dim=-1).item()

    return "Toxic" if pred_idx == 1 else "Non-Toxic"

# Now this will work:
print(predict("你好"))

In [ ]:
predict("你好")

In [ ]:
# 11. Confusion Matrix & Classification Report
predictions = trainer.predict(tokenized_dataset["validation"])
y_true = predictions.label_ids
y_pred = np.argmax(predictions.predictions, axis=-1)

print("\nClassification Report:")
print(classification_report(y_true, y_pred, labels=[0, 1], target_names=["Non-Toxic", "Toxic"]))

In [ ]:
# Plot confusion matrix
import matplotlib.pyplot as plt
import seaborn as sns
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Non-Pun", "Pun"],
            yticklabels=["Non-Pun", "Pun"],
            cbar_kws={'label': 'Count'})
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix - Pun Classification")
plt.tight_layout()
plt.show()

print("\n✅ Model training completed!")